## Agentic RAG - Agent가 검색 전략 자율 결정

이전 노트북([01_Graph RAG](01_Graph%20RAG%20-%20관계형%20검색%20원리.ipynb))에서는 "언제 벡터 검색을 쓰고
언제 그래프 순회를 쓸지", "몇 홉까지 순회할지"를 모두 사람이 코드에 **고정값**으로 박아 두었다. 실제로는
질문마다 필요한 전략이 다르다.

- 인사말이나 일반 상식 질문 → 검색 자체가 필요 없음
- "OO은 어느 팀이야?" 같은 단순 사실 조회 → 벡터 검색 한 번이면 충분
- "A가 담당하는 B가 의존하는 C는 누가 담당해?" 같은 멀티홉 질문 → 그래프 순회 필요
- 질문에 쓰인 표현이 그래프의 개체명과 정확히 안 맞을 때 → 먼저 벡터 검색으로 정확한 개체명을 찾은 뒤 그래프로 넘어가야 함
- 복합 질문 → 벡터 검색과 그래프 순회를 여러 번 조합해야 함

이 노트북에서는 벡터 검색과 그래프 검색을 각각 **도구(tool)**로 만들어 LLM에게 쥐여주고, "어떤 도구를 쓸지",
"몇 번 쓸지", "언제 검색을 멈추고 답할지"를 **Agent가 스스로 판단**하도록 LangGraph로 구성한다.
고정 파이프라인(Naive RAG, 고정 Graph RAG)과 결과를 비교해 자율 검색 에이전트가 실제로 무엇을 얻는지 확인한다.

In [1]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
load_dotenv("C:/env/.env")

True

### [0] 공통 준비: LLM, 임베딩 모델

In [2]:
from typing import Annotated, List, TypedDict

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

c:\Users\storm\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22392\346001978.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### [1] 샘플 도메인 재구성: 테크노바 지식베이스

01번 노트북과 동일한 "테크노바" 조직·프로젝트 사실 문장을 그대로 사용한다. 벡터 스토어와 지식 그래프를
다시 만들어 이 노트북만으로도 독립 실행이 되도록 한다. (트리플 추출·그래프 구축 원리 자체는 01번 노트북 참고)

In [3]:
company_facts = [
    "김민준은 테크노바의 AI팀 소속이다.",
    "이서연은 AI팀의 팀장이다.",
    "AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.",
    "박지훈은 데이터팀 소속이다.",
    "최유진은 데이터팀의 팀장이다.",
    "데이터팀은 '추천시스템 고도화' 프로젝트를 담당한다.",
    "AI팀은 데이터팀과 긴밀히 협업한다.",
    "정다은은 인프라팀 소속이다.",
    "한소희는 인프라팀의 팀장이다.",
    "인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.",
    "데이터팀은 인프라팀과 긴밀히 협업한다.",
    "'그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.",
    "박지훈은 '추천시스템 고도화' 프로젝트의 담당자이다.",
    "김민준은 '그래프 RAG 엔진' 프로젝트의 담당자이다.",
    "프로덕트팀은 '온보딩 자동화' 프로젝트를 담당한다.",
    "정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.",
]

docs = [
    Document(page_content=fact, metadata={"fact_id": i})
    for i, fact in enumerate(company_facts)
]

vectorstore = FAISS.from_documents(docs, embeddings)


def format_docs(documents: List[Document]) -> str:
    return "\n".join(f"- {d.page_content}" for d in documents)


print(f"사실 문장 수: {len(company_facts)}")

사실 문장 수: 16


In [4]:
from pydantic import BaseModel, Field


class Triple(BaseModel):
    subject: str = Field(description="관계의 주체가 되는 개체명")
    relation: str = Field(
        description="MEMBER_OF, LEADS, OWNS, COLLABORATES_WITH, DEPENDS_ON, WORKS_ON 중 하나"
    )
    object: str = Field(description="관계의 대상이 되는 개체명")


class TripleList(BaseModel):
    triples: List[Triple]


triple_extractor = llm.with_structured_output(TripleList)

extract_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "다음 문장들에서 (주체, 관계, 객체) 트리플을 모두 추출하라.\n"
            "- relation은 반드시 MEMBER_OF, LEADS, OWNS, COLLABORATES_WITH, DEPENDS_ON, WORKS_ON 중 하나로 매핑한다.\n"
            "- 개체명은 문장에 등장한 표기를 그대로 사용하고, 작은따옴표는 제거한다.\n"
            "- 한 문장에 여러 관계가 있으면 모두 추출한다.",
        ),
        ("human", "{text}"),
    ]
)
extract_chain = extract_prompt | triple_extractor

extracted = extract_chain.invoke({"text": "\n".join(company_facts)})
triples = [(t.subject, t.relation, t.object) for t in extracted.triples]

print(f"추출된 트리플 수: {len(triples)}")

추출된 트리플 수: 16


In [5]:
import networkx as nx

graph = nx.MultiDiGraph()
for subject, relation, obj in triples:
    graph.add_edge(subject, obj, relation=relation)


def k_hop_edges(g: nx.MultiDiGraph, seeds: List[str], k: int = 2) -> List[tuple]:
    '''seed로부터 양방향으로 최대 k홉까지 순회하며 지나온 엣지를 수집한다.'''
    visited_nodes = set(seeds)
    frontier = set(seeds)
    collected_edges = set()

    for _ in range(k):
        next_frontier = set()
        for node in frontier:
            for _, neighbor, data in g.out_edges(node, data=True):
                collected_edges.add((node, data["relation"], neighbor))
                if neighbor not in visited_nodes:
                    next_frontier.add(neighbor)
            for neighbor, _, data in g.in_edges(node, data=True):
                collected_edges.add((neighbor, data["relation"], node))
                if neighbor not in visited_nodes:
                    next_frontier.add(neighbor)
        visited_nodes |= next_frontier
        frontier = next_frontier
        if not frontier:
            break

    return sorted(collected_edges)


def edges_to_context(edges: List[tuple]) -> str:
    return "\n".join(f"- ({s}) -[{r}]-> ({o})" for s, r, o in edges)


print(f"노드 수: {graph.number_of_nodes()}, 엣지 수: {graph.number_of_edges()}")

노드 수: 15, 엣지 수: 16


### [2] 검색 도구 정의: 벡터 검색 vs 그래프 검색

각 도구의 **설명(docstring)**이 곧 Agent가 도구를 선택하는 기준이 된다. 설명을 명확히 적어야 LLM이
질문 유형에 맞는 도구를 고를 수 있다.

- `vector_search`: 질문과 의미적으로 유사한 사실 문장을 찾는다. 단일 사실 조회에 적합하다.
- `graph_search`: 개체명을 기점으로 지식 그래프를 순회해 관계를 추적한다. 개체명이 정확히 일치해야
  탐색이 가능하며, 못 찾으면 그 사실을 그대로 반환해 Agent가 다른 전략(예: 벡터 검색으로 정확한 이름 찾기)을
  시도하도록 유도한다.

In [6]:
from langchain_core.tools import tool


@tool
def vector_search(query: str) -> str:
    '''질문과 의미적으로 유사한 사실 문장을 벡터 유사도로 검색한다.
    "OO은 어느 팀이야?", "OO 프로젝트 담당자는?"처럼 단일 사실을 조회할 때 적합하다.
    정확한 개체명(고유명사)을 모를 때 실마리를 찾는 용도로도 쓸 수 있다.'''
    retrieved = vectorstore.similarity_search(query, k=4)
    return format_docs(retrieved) if retrieved else "검색 결과 없음"


@tool
def graph_search(entity: str, hops: int = 2) -> str:
    '''지식 그래프에서 entity(개체명)를 시작점으로 최대 hops단계까지 관계를 순회해 조회한다.
    "A가 담당하는 B가 의존하는 C는?"처럼 여러 단계를 연결해야 하는 멀티홉 질문에 적합하다.
    entity는 그래프 노드명과 정확히(또는 부분적으로) 일치해야 하며, 일치하는 노드가 없으면 찾지
    못했다고 반환하므로 이 경우 vector_search로 정확한 개체명을 먼저 확인해야 한다.'''
    seeds = [node for node in graph.nodes() if entity in node or node in entity]
    if not seeds:
        return f"'{entity}'와 일치하는 개체를 그래프에서 찾지 못함. 정확한 개체명 확인이 필요하다."
    edges = k_hop_edges(graph, seeds, k=hops)
    return edges_to_context(edges) if edges else f"'{entity}'에서 시작하는 관계를 찾지 못함"


tools = [vector_search, graph_search]

### [3] LangGraph로 자율 검색 루프 구성

흐름은 단순한 ReAct 루프다.

1. `agent` 노드: LLM이 지금까지의 대화(질문 + 이전 검색 결과)를 보고 **도구를 호출할지, 바로 답할지**를 스스로 결정한다.
2. 도구 호출을 요청하면 `tools` 노드가 실제로 도구를 실행하고 결과를 `ToolMessage`로 되돌려준다.
3. `agent` 노드가 다시 그 결과를 보고 **검색을 더 할지, 충분하니 답할지**를 판단한다. 도구 호출이 없으면 종료.

몇 번 검색할지, 어떤 도구를 쓸지에 대한 고정 로직은 없다. 전부 시스템 프롬프트의 지침과 LLM의 판단에 맡긴다.

In [7]:
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], add_messages]


SYSTEM_PROMPT = """당신은 테크노바 사내 지식 어시스턴트다. 다음 두 검색 도구를 자유롭게 사용할 수 있다.

- vector_search: 의미 기반 유사도 검색. 단일 사실 조회, 정확한 개체명을 모를 때 실마리 찾기에 적합.
- graph_search: 개체명 기점 그래프 순회. 여러 단계를 연결해야 하는 멀티홉 관계 질문에 적합.

지침:
1. 검색 없이 답할 수 있는 질문(인사, 일반 지식)은 도구를 호출하지 말고 바로 답하라.
2. 어떤 도구가 적합한지, 몇 번 호출할지는 스스로 판단하라.
3. graph_search가 개체를 찾지 못하면, vector_search로 정확한 개체명을 먼저 파악한 뒤 다시 graph_search를 시도하라.
4. 충분한 근거를 모았다고 판단되면 검색을 멈추고 근거에 기반해 답하라. 근거를 다 찾아봐도 부족하면 모른다고 답하라.
5. 최종 답변에는 근거로 사용한 사실/관계를 간단히 함께 제시하라."""

llm_with_tools = llm.bind_tools(tools)


def agent_node(state: AgentState) -> dict:
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tools"
    return END


workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", ToolNode(tools))
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")

agentic_rag = workflow.compile()

print(agentic_rag.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### [4] 실행 도우미: 검색 트레이스 출력

Agent가 실제로 어떤 도구를, 어떤 입력으로, 몇 번 호출했는지 순서대로 확인할 수 있게 트레이스를 출력한다.

In [8]:
def run_agentic_rag(question: str, verbose: bool = True) -> str:
    result = agentic_rag.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"recursion_limit": 15},
    )
    messages = result["messages"]

    if verbose:
        print(f"질문: {question}\n")
        step = 1
        for m in messages[1:]:
            if getattr(m, "tool_calls", None):
                for call in m.tool_calls:
                    print(f"[{step}] Agent 판단 → 도구 호출: {call['name']}({call['args']})")
                    step += 1
            elif m.type == "tool":
                preview = m.content if len(m.content) < 300 else m.content[:300] + " ..."
                print(f"    └─ 도구 결과:\n{preview}\n")
            elif m.type == "ai" and m.content:
                print(f"[{step}] Agent 최종 답변:\n{m.content}")

    return messages[-1].content

### [5] 실험 1 — 검색이 필요 없는 질문

인사말에는 도구를 호출하지 않고 바로 답하는지 확인한다.

In [9]:
_ = run_agentic_rag("안녕? 너는 무슨 일을 도와줄 수 있어?")

질문: 안녕? 너는 무슨 일을 도와줄 수 있어?

[1] Agent 최종 답변:
안녕하세요! 저는 테크노바 사내 지식 어시스턴트로, 회사와 관련된 정보나 질문에 대해 도움을 드릴 수 있습니다. 예를 들어, 프로젝트 정보, 팀 구성, 특정 기술에 대한 질문 등을 처리할 수 있습니다. 필요한 정보가 있으면 말씀해 주세요!


### [6] 실험 2 — 단순 사실 조회

단일 사실만 있으면 답이 되는 질문이므로 `vector_search`를 한 번만 호출하고 끝내는지 확인한다.

In [13]:
_ = run_agentic_rag("이서연은 어느 팀 팀장이야?")

질문: 이서연은 어느 팀 팀장이야?

[1] Agent 판단 → 도구 호출: vector_search({'query': '이서연 팀장'})
    └─ 도구 결과:
- 이서연은 AI팀의 팀장이다.
- 한소희는 인프라팀의 팀장이다.
- 최유진은 데이터팀의 팀장이다.
- 정다은은 인프라팀 소속이다.

[2] Agent 최종 답변:
이서연은 AI팀의 팀장입니다.


### [7] 실험 3 — 멀티홉 관계 질문

01번 노트북에서 Naive Vector RAG가 답하지 못했던 질문이다. Agent가 `graph_search`를 선택하는지 확인한다.

In [16]:
_ = run_agentic_rag("김민준이 담당하는 프로젝트가 의존하는 프로젝트는 누가 담당해?")

질문: 김민준이 담당하는 프로젝트가 의존하는 프로젝트는 누가 담당해?

[1] Agent 판단 → 도구 호출: graph_search({'entity': '김민준'})
    └─ 도구 결과:
- (AI팀) -[WORKS_ON]-> (그래프 RAG 엔진)
- (그래프 RAG 엔진) -[DEPENDS_ON]-> (검색 인프라 개선)
- (김민준) -[MEMBER_OF]-> (테크노바의 AI팀)
- (김민준) -[WORKS_ON]-> (그래프 RAG 엔진)

[2] Agent 판단 → 도구 호출: graph_search({'entity': '검색 인프라 개선'})
    └─ 도구 결과:
- (AI팀) -[WORKS_ON]-> (그래프 RAG 엔진)
- (그래프 RAG 엔진) -[DEPENDS_ON]-> (검색 인프라 개선)
- (김민준) -[WORKS_ON]-> (그래프 RAG 엔진)
- (데이터팀) -[COLLABORATES_WITH]-> (인프라팀)
- (인프라팀) -[WORKS_ON]-> (검색 인프라 개선)
- (정다은) -[MEMBER_OF]-> (인프라팀)
- (정다은) -[WORKS_ON]-> (검색 인프라 개선)
- (한소희) -[LEADS]-> (인프라팀)

[3] Agent 최종 답변:
김민준이 담당하는 프로젝트인 "그래프 RAG 엔진"은 "검색 인프라 개선" 프로젝트에 의존하고 있습니다. 이 "검색 인프라 개선" 프로젝트는 정다은이 담당하고 있습니다.

**근거:**
- 김민준은 "그래프 RAG 엔진" 프로젝트를 담당하고 있으며, 이 프로젝트는 "검색 인프라 개선"에 의존하고 있습니다.
- "검색 인프라 개선" 프로젝트는 정다은이 담당하고 있습니다.


### [8] 실험 4 — 첫 시도 실패 후 전략 전환

질문에 쓰인 표현("AI 관련 프로젝트")이 그래프의 정확한 노드명("그래프 RAG 엔진")과 다르다.
`graph_search`를 먼저 시도했다가 개체를 찾지 못하면, Agent가 스스로 `vector_search`로 정확한 이름을
파악한 뒤 다시 `graph_search`를 시도하는지 확인한다.

In [17]:
_ = run_agentic_rag("우리 회사 AI 관련 프로젝트를 담당하는 팀은 어떤 팀과 협업해?")

질문: 우리 회사 AI 관련 프로젝트를 담당하는 팀은 어떤 팀과 협업해?

[1] Agent 판단 → 도구 호출: vector_search({'query': 'AI 관련 프로젝트 담당 팀 협업'})
    └─ 도구 결과:
- AI팀은 데이터팀과 긴밀히 협업한다.
- AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
- 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
- 데이터팀은 '추천시스템 고도화' 프로젝트를 담당한다.

[2] Agent 판단 → 도구 호출: graph_search({'entity': 'AI팀'})
    └─ 도구 결과:
- (AI팀) -[COLLABORATES_WITH]-> (데이터팀)
- (AI팀) -[WORKS_ON]-> (그래프 RAG 엔진)
- (그래프 RAG 엔진) -[DEPENDS_ON]-> (검색 인프라 개선)
- (김민준) -[MEMBER_OF]-> (테크노바의 AI팀)
- (김민준) -[WORKS_ON]-> (그래프 RAG 엔진)
- (데이터팀) -[COLLABORATES_WITH]-> (인프라팀)
- (데이터팀) -[WORKS_ON]-> (추천시스템 고도화)
- (박지훈) -[MEMBER_OF]-> (데이터팀)
- (이서연) -[ ...

[3] Agent 최종 답변:
우리 회사의 AI 관련 프로젝트를 담당하는 팀은 **AI팀**입니다. 이 팀은 **데이터팀**과 긴밀히 협업하고 있습니다. AI팀은 '그래프 RAG 엔진' 프로젝트를 진행하고 있으며, 데이터팀은 '추천시스템 고도화' 프로젝트를 담당하고 있습니다. 

근거:
- AI팀은 데이터팀과 협업한다는 정보.
- AI팀이 '그래프 RAG 엔진' 프로젝트를 담당하고 있다는 정보.


### [9] 실험 5 — 여러 도구를 조합해야 하는 복합 질문

In [18]:
_ = run_agentic_rag("정다은이 담당하는 프로젝트는 무엇이고, 그 프로젝트에 의존하는 프로젝트를 담당하는 사람은 누구야?")

질문: 정다은이 담당하는 프로젝트는 무엇이고, 그 프로젝트에 의존하는 프로젝트를 담당하는 사람은 누구야?

[1] Agent 판단 → 도구 호출: graph_search({'entity': '정다은'})
    └─ 도구 결과:
- (그래프 RAG 엔진) -[DEPENDS_ON]-> (검색 인프라 개선)
- (데이터팀) -[COLLABORATES_WITH]-> (인프라팀)
- (인프라팀) -[WORKS_ON]-> (검색 인프라 개선)
- (정다은) -[MEMBER_OF]-> (인프라팀)
- (정다은) -[WORKS_ON]-> (검색 인프라 개선)
- (한소희) -[LEADS]-> (인프라팀)

[2] Agent 판단 → 도구 호출: graph_search({'entity': '검색 인프라 개선'})
    └─ 도구 결과:
- (AI팀) -[WORKS_ON]-> (그래프 RAG 엔진)
- (그래프 RAG 엔진) -[DEPENDS_ON]-> (검색 인프라 개선)
- (김민준) -[WORKS_ON]-> (그래프 RAG 엔진)
- (데이터팀) -[COLLABORATES_WITH]-> (인프라팀)
- (인프라팀) -[WORKS_ON]-> (검색 인프라 개선)
- (정다은) -[MEMBER_OF]-> (인프라팀)
- (정다은) -[WORKS_ON]-> (검색 인프라 개선)
- (한소희) -[LEADS]-> (인프라팀)

[3] Agent 최종 답변:
정다은은 "검색 인프라 개선" 프로젝트를 담당하고 있습니다. 이 프로젝트에 의존하는 "그래프 RAG 엔진" 프로젝트는 김민준이 담당하고 있습니다.

**근거:**
1. 정다은은 인프라팀의 일원으로 "검색 인프라 개선" 프로젝트에 참여하고 있습니다.
2. "검색 인프라 개선" 프로젝트는 "그래프 RAG 엔진"에 의존하고 있으며, 이 프로젝트는 김민준이 담당하고 있습니다.


### [10] Fixed RAG vs Agentic RAG 비교

같은 질문 5개를 (a) 항상 벡터 검색만 쓰는 Naive RAG, (b) 항상 그래프 검색만 쓰는 Fixed Graph RAG,
(c) Agent가 전략을 스스로 고르는 Agentic RAG에 각각 던져 도구 호출 패턴을 비교한다.

In [19]:
naive_answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "아래 [사실] 목록만 근거로 질문에 답하라. 목록에 없는 내용은 추측하지 말고 "
            "'주어진 사실만으로는 알 수 없다'라고 답하라.\n\n[사실]\n{context}",
        ),
        ("human", "{question}"),
    ]
)
naive_chain = naive_answer_prompt | llm

graph_answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "아래 [관계] 목록은 지식 그래프에서 순회로 수집한 사실이다. 이 관계들을 연결해 질문에 답하라. "
            "목록만으로 답할 수 없으면 '알 수 없다'라고 답하라.\n\n[관계]\n{context}",
        ),
        ("human", "{question}"),
    ]
)
graph_chain = graph_answer_prompt | llm


def naive_rag(question: str) -> str:
    retrieved = vectorstore.similarity_search(question, k=4)
    return naive_chain.invoke({"context": format_docs(retrieved), "question": question}).content


def fixed_graph_rag(question: str) -> str:
    seeds = [node for node in graph.nodes() if node in question]
    edges = k_hop_edges(graph, seeds, k=3) if seeds else []
    return graph_chain.invoke({"context": edges_to_context(edges), "question": question}).content


comparison_queries = [
    "안녕? 너는 무슨 일을 도와줄 수 있어?",
    "이서연은 어느 팀 팀장이야?",
    "김민준이 담당하는 프로젝트가 의존하는 프로젝트는 누가 담당해?",
    "우리 회사 AI 관련 프로젝트를 담당하는 팀은 어떤 팀과 협업해?",
    "정다은이 담당하는 프로젝트는 무엇이고, 그 프로젝트에 의존하는 프로젝트를 담당하는 사람은 누구야?",
]

for q in comparison_queries:
    naive = naive_rag(q)
    fixed_graph = fixed_graph_rag(q)
    agentic = run_agentic_rag(q, verbose=False)

    print(f"질문: {q}")
    print(f"- Naive Vector RAG (항상 벡터 검색): {naive}")
    print(f"- Fixed Graph RAG (항상 그래프 검색): {fixed_graph}")
    print(f"- Agentic RAG (전략 자율 결정)     : {agentic}")
    print("-" * 80)

질문: 안녕? 너는 무슨 일을 도와줄 수 있어?
- Naive Vector RAG (항상 벡터 검색): 주어진 사실만으로는 알 수 없다.
- Fixed Graph RAG (항상 그래프 검색): 안녕하세요! 저는 다양한 질문에 답변하고, 정보 제공, 문제 해결, 글쓰기 도움 등 여러 가지 일을 도와줄 수 있습니다. 궁금한 점이 있으면 말씀해 주세요!
- Agentic RAG (전략 자율 결정)     : 안녕하세요! 저는 테크노바 사내 지식 어시스턴트로, 회사와 관련된 정보나 질문에 대해 도움을 드릴 수 있습니다. 예를 들어, 프로젝트 정보, 팀 구성, 특정 기술에 대한 질문 등을 처리할 수 있습니다. 필요한 정보가 있으면 말씀해 주세요!
--------------------------------------------------------------------------------
질문: 이서연은 어느 팀 팀장이야?
- Naive Vector RAG (항상 벡터 검색): 이서연은 AI팀의 팀장이다.
- Fixed Graph RAG (항상 그래프 검색): 이서연은 AI팀의 팀장이다.
- Agentic RAG (전략 자율 결정)     : 이서연은 AI팀의 팀장입니다.
--------------------------------------------------------------------------------
질문: 김민준이 담당하는 프로젝트가 의존하는 프로젝트는 누가 담당해?
- Naive Vector RAG (항상 벡터 검색): 주어진 사실만으로는 알 수 없다.
- Fixed Graph RAG (항상 그래프 검색): 김민준이 담당하는 프로젝트는 그래프 RAG 엔진이며, 이 엔진은 검색 인프라 개선에 의존하고 있습니다. 검색 인프라 개선은 인프라팀과 정다은이 담당하고 있습니다.
- Agentic RAG (전략 자율 결정)     : 김민준이 담당하는 프로젝트인 "그래프 RAG 엔진"은 "검색 인프라 개선" 프로젝트에 의존하고 있습니다. 이 "검색 인프라 개선" 

### [11] 정리

| 구분 | Naive Vector RAG | Fixed Graph RAG | Agentic RAG |
|---|---|---|---|
| 검색 여부 결정 | 항상 검색 | 항상 검색 | 질문에 따라 검색 생략 가능 |
| 검색 전략 선택 | 고정(벡터만) | 고정(그래프만) | 질문 유형에 맞춰 벡터/그래프를 LLM이 선택 |
| 검색 횟수 | 1회 고정 | 1회 고정(k홉 고정) | 결과가 불충분하면 재검색·전략 전환 (가변) |
| 실패 시 대응 | 없음 (top-k 안에 없으면 끝) | 없음 (개체 못 찾으면 끝) | 다른 도구/질의로 재시도 |
| 비용·지연 | 낮음 | 낮음 | LLM 호출·도구 호출 반복으로 더 높음 |

**구현 포인트**
- 검색 로직을 `@tool`로 감싸고, **도구 설명(docstring)**을 명확히 써서 Agent가 도구를 스스로 선택할 근거를 준다.
- LangGraph의 `agent → tools → agent → ...` 루프(ReAct 패턴)로 "검색할지 말지", "몇 번 검색할지"를
  하드코딩하지 않고 LLM 판단에 맡긴다.
- `graph_search`가 실패 이유를 텍스트로 반환하도록 만들어, Agent가 실패를 보고 **다른 전략으로 전환**하도록 유도했다.
  (도구 결과가 명확할수록 Agent의 다음 판단이 좋아진다.)
- `recursion_limit`으로 무한 루프를 방지해 비용을 통제한다.

**한계와 실전 확장 포인트**
- 매 단계 LLM 호출이 필요해 고정 파이프라인보다 느리고 비용이 크다. 질문 유형이 뻔한 서비스라면 오히려 고정 라우팅이 나을 수 있다.
- 이 노트북은 도구 결과를 그대로 신뢰하지만, **Self-RAG/CRAG**처럼 검색 결과의 관련성을 LLM이 별도로
  채점(grading)하고 낮으면 질의를 재작성(query rewriting)하는 단계를 추가하면 더 견고해진다.
- 웹 검색, SQL 조회 등 다른 도구를 추가하면 Agent가 선택할 수 있는 전략의 폭이 더 넓어진다.
- 그래프 노드명이 질문 표현과 자주 어긋난다면, 문자열 매칭 대신 임베딩 기반 엔티티 링킹으로 `graph_search`의
  개체 인식 자체를 보강하는 것도 방법이다.